# 6 post engagement

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections
# Rounded event-week mean engagement, as calculated in the original notebook.
post_engage = pd.read_parquet('data/post_engagement_weekly.parquet')
print(post_engage.shape)


In [ ]:
source_corrections = get_source_corrections(window_days=28).query('prior_corrections==0').reset_index(drop=True)
source_corrections_matched = pd.read_parquet('data/source_corrections_content_matched.parquet')
source_corrections = source_corrections.merge(source_corrections_matched[['tweet_id']], on='tweet_id')
print(source_corrections.shape)
source_corrections.head()

In [ ]:
post_engage = post_engage.merge(source_corrections, on='tweet_id')
print(post_engage.shape)
post_engage.head()

In [ ]:
from utils import StataDiD
stata_did = StataDiD(post_engage, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

### Retweets

In [ ]:
from utils import get_coefs
stata_did.multi_did(reg='qui ppmlhdfe', dev='user_tweet_retweets', cluster='account_cluster')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=0, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='user_tweet_retweets', cluster='account_cluster')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=0, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-1.862, 3.262)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Change in engagement\n(reposts per post)', 'did_reposts_weekly', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Replies

In [ ]:
stata_did.multi_did(reg='qui ppmlhdfe', dev='user_tweet_replies', cluster='account_cluster')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=0, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='user_tweet_replies', cluster='account_cluster')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=0, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Change in engagement\n(replies per post)', 'did_replies_weekly', ylims=ylims, legend_ncol=1, legend_title='First correction')